# Patient 360 Table

Documentation: Add links to word doc here.


## Version Control
| Version      | Description                                      |
|--------------|--------------------------------------------------|
|v5 (01/16)| Make changes to the timewindows, making end dates dynamic|
| v4 (01/07)   |Added newborn_screening_flag based on patient state list (incl. CT). Built latest Place of Service fields by deriving most_recent_infusion_location from most recent qualifying Tx event and mapping to most_recent_infusion_description; blank POS values are standardized to NULL. |
| v3 (12/26)   | Reference File changes flowing into patient 360 now (Komodo affiliation integrated, Julie's mapping file in effect for hco npis, hco_zips, territory, region have been updated, region id and territory id has been added) |
| v2 (12/23)   | Added age_bucket, zip3, first_tx_after_diagnosis, last_mpsii_tx_date, latest treatment type, time cols, payer cols, elaprase fills (Section A) |
| v1 (12/18)   | Primary HCP affiliation logic modified and validated. |


## Time Window
- **Treatment Window:** August 1, 2023 - November 30, 2025 
- **Diagnosis Lookback:** August 1, 2020 - November 30, 2025

## MPSII Diagnosis 1:1 HCP Mapping Query

## Patient Eligibility
- **Specified diagnosis (E761):** 2+ diagnosis claims in 5-year window
- **Unspecified diagnosis (E763):** 2+ diagnosis claims in 5-year window (not already counted in specified)
- **Both:** Must have Elaprase treatment (NDC or procedure codes)

## Ranking Tiers (applied in order)

### 1. Specialty Priority
```
Geneticist (1) > 
Psychiatry & Neurology (2) > 
Pediatrician (3) > 
PCP (4) > 
NPPA (5) > 
Others (6) > 
NA (7)
```

### 2. Visit Count
Higher number of distinct treatment dates = better rank

### 3. Recency
Most recent treatment date = better rank

### 4. NPI Tiebreaker
Lower NPI value = wins

## Output Columns

| Column | Description |
|--------|-------------|
| PATIENT_ID | Patient identifier |
| FINAL_NPI | Assigned HCP provider number |
| SPECIALTY | Classified specialty type |
| SPECIALTY_PRIORITY | Priority rank (1-7) |
| NO_OF_VISITS | Count of distinct visit dates |
| MOST_RECENT_VISIT | Date of most recent visit |
| HCP_RANK | Ranking within patient (always 1 in final output) |
| PATIENT_TYPE | Always 'DX' for diagnosis patients |

## Specialty Classification
- **Geneticist:** Primary or secondary specialty contains "Genetic"
- **Psychiatry & Neurology:** Contains "Psychiatry & Neurology" or "Neurological Surgery" or secondary contains "Neurodevelopmental Disabilities"
- **Pediatrician:** Primary specialty contains "Pediatrics"
- **PCP:** Primary or secondary contains "Internal Medicine" or "Family Medicine"
- **NPPA:** Primary specialty contains "Nurse Practitioner" or "Physician Assistant"
- **NA:** NPI is NULL
- **Others:** Everything else

## Data Sources
- **kom_medical_events:** Medical claims (NDC, procedure codes, diagnoses)
- **kom_pharmacy_events:** Pharmacy claims (NDC codes, diagnoses, paid status)
- **kom_providers:** HCP specialties

## Filtering Criteria
- **Pharmacy claims:** TRANSACTION_STATUS = 'PAID' only
- **Diagnosis codes:** E761 (specified), E763 (unspecified)
- **Treatment NDC:** 54092070001, 540920700 (Elaprase)
- **Treatment Procedures:** J1743, 99601, 99602, 96365, 96366, S9357, S9379, 38206-38250

In [0]:
CREATE WIDGET TEXT end_date DEFAULT '2025-11-30';

In [0]:
%sql
-- =============================================================================
-- MPSII Diagnosis Patients 1:1 HCP Mapping with 4-Tier Preference Logic
-- =============================================================================
-- Treatment Window: 2023-08-01 to 2025-11-30 (2 years)
-- Diagnosis Lookback: 2020-08-01 to 2025-11-30 (5 years)
-- 
-- Preference Logic:
-- 1. Specialty Priority (Geneticist > Psych & Neuro > Pediatrician > PCP > NPPA > Others)
-- 2. Number of Visits (COUNT DISTINCT FILL_DATE) - Higher is better
-- 3. Most Recent Visit (MAX FILL_DATE) - More recent is better
-- 4. NPI as Tiebreaker - Lower NPI wins
-- =============================================================================

-- Step 1: Create Treatment Table (2 years)
CREATE OR REPLACE TEMPORARY VIEW mpsii_treatment_table AS
SELECT DISTINCT *
FROM (
    -- Medical Events - NDC Codes
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        NDC11 AS CODE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')
    
    UNION
    
    -- Pharmacy Events - NDC Codes
    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        NDC11 AS CODE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        FILL_DATE,
        NULL AS PLACE_OF_SERVICE,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        'PHARMACY_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
    
    UNION
    
    -- Medical Events - Procedure Codes
    SELECT DISTINCT 
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        PROCEDURE_CODE AS CODE,   
        MEDICAL_EVENT_ID AS EVENT_ID,
        SERVICE_DATE AS FILL_DATE, 
        PLACE_OF_SERVICE,
        KH_PLAN_ID AS KH_PLAN,
        'MEDICAL_EVENTS' AS TABLE_NAME
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                            '38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';


-- Step 2: Create Diagnosis Table with Patient Eligibility Logic
CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_table AS 
WITH mpsii_1dx_specified AS (
    SELECT DISTINCT * FROM (
        -- Medical Events - Specified Diagnosis
        SELECT DISTINCT 
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            Place_of_service
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'

        UNION

        -- Pharmacy Events - Specified Diagnosis
        SELECT DISTINCT 
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS Place_of_service
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
            AND TRANSACTION_STATUS = 'PAID'
    ) AS combined
    WHERE FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
mpsii_2dx_specified AS (
    SELECT *
    FROM mpsii_1dx_specified
    WHERE patient_id IN (
        SELECT a.patient_id 
        FROM mpsii_1dx_specified AS a 
        GROUP BY a.patient_id 
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    )
),
mpsii_1dx_unspecified AS (
    SELECT DISTINCT * FROM (
        -- Medical Events - Unspecified Diagnosis
        SELECT DISTINCT 
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            SERVICE_DATE AS FILL_DATE,
            MEDICAL_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODES,
            KH_PLAN_ID AS KH_PLAN,
            Place_of_service
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'

        UNION

        -- Pharmacy Events - Unspecified Diagnosis
        SELECT DISTINCT 
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            FILL_DATE,
            PHARMACY_EVENT_ID AS EVENT_ID,
            DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
            NULL AS Place_of_service
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
            AND TRANSACTION_STATUS = 'PAID'
    ) AS combined
    WHERE FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
mpsii_2dx_unspecified AS (
    SELECT DISTINCT *
    FROM mpsii_1dx_unspecified
    WHERE patient_id IN (
        SELECT DISTINCT a.patient_id 
        FROM mpsii_1dx_unspecified AS a 
        GROUP BY a.patient_id 
        HAVING COUNT(DISTINCT a.fill_date) >= 2
    ) 
),
mpsii_2dx_specified_tx AS (
    SELECT DISTINCT *
    FROM mpsii_treatment_table 
    WHERE patient_id IN (
        SELECT DISTINCT a.patient_id 
        FROM mpsii_2dx_specified AS a
    )
),
incremental_patient AS (
    SELECT DISTINCT patient_id
    FROM mpsii_2dx_unspecified
    WHERE patient_id IN (
        SELECT DISTINCT a.patient_id 
        FROM mpsii_treatment_table AS a 
        WHERE a.code IN ('54092070001','540920700','J1743')
    )
    AND patient_id NOT IN (
        SELECT DISTINCT b.patient_id 
        FROM mpsii_2dx_specified_tx AS b
    )
),
all_dx_patients_claims AS (
    -- All specified 2Dx patients
    SELECT DISTINCT * FROM mpsii_2dx_specified
    
    UNION
    
    -- Incremental unspecified patients with Elaprase treatment
    SELECT DISTINCT * 
    FROM mpsii_1dx_unspecified 
    WHERE patient_id IN (
        SELECT DISTINCT a.patient_id 
        FROM incremental_patient AS a
    ) 
)
SELECT DISTINCT *
FROM all_dx_patients_claims;


-- Step 3: Apply 1:1 Mapping Logic with 4-Tier Preference Conditions
CREATE OR REPLACE TEMPORARY VIEW Dx_1_1_Mapped_HCP_Final AS
SELECT 
    PATIENT_ID,
    NPI AS FINAL_NPI,
    SPECIALTY,
    PRIORITY AS SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK,
    'DX' AS PATIENT_TYPE
FROM (
    SELECT 
        PATIENT_ID,
        NPI,
        SPECIALTY,
        PRIORITY,
        NO_OF_VISITS,
        MOST_RECENT_VISIT,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                PRIORITY ASC,           -- Condition 1: Specialty (lower number = higher priority)
                NO_OF_VISITS DESC,      -- Condition 2: More visits = better
                MOST_RECENT_VISIT DESC, -- Condition 3: More recent = better
                NPI ASC                 -- Condition 4: Lower NPI wins tiebreaker
        ) AS HCP_RANK
    FROM (
        SELECT 
            a.PATIENT_ID,
            a.NPI,
            -- Specialty Classification
            CASE 
                WHEN p.primary_specialty LIKE '%Genetic%' OR p.secondary_specialty LIKE '%Genetic%' 
                    THEN 'Geneticist'
                WHEN p.primary_specialty LIKE '%Pediatrics%' 
                    THEN 'Pediatrician'
                WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' OR 
                     p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                     p.primary_specialty LIKE '%Neurological Surgery%' 
                    THEN 'Psychiatry & Neurology'
                WHEN p.primary_specialty LIKE '%Nurse Practitioner%' OR 
                     p.primary_specialty LIKE '%Physician Assistant%' 
                    THEN 'NPPA'
                WHEN p.primary_specialty LIKE '%Internal Medicine%' OR 
                     p.secondary_specialty LIKE '%Internal Medicine%' 
                    THEN 'PCP'
                WHEN p.primary_specialty LIKE '%Family Medicine%' OR 
                     p.secondary_specialty LIKE '%Family Medicine%' 
                    THEN 'PCP'
                WHEN a.NPI IS NULL 
                    THEN 'NA'
                ELSE 'Others'
            END AS SPECIALTY,
            -- Priority Ranking (1 = Highest Priority)
            CASE 
                WHEN p.primary_specialty LIKE '%Genetic%' OR p.secondary_specialty LIKE '%Genetic%' 
                    THEN 1
                WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' OR 
                     p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR 
                     p.primary_specialty LIKE '%Neurological Surgery%' 
                    THEN 2
                WHEN p.primary_specialty LIKE '%Pediatrics%' 
                    THEN 3
                WHEN p.primary_specialty LIKE '%Internal Medicine%' OR 
                     p.secondary_specialty LIKE '%Internal Medicine%' OR
                     p.primary_specialty LIKE '%Family Medicine%' OR 
                     p.secondary_specialty LIKE '%Family Medicine%' 
                    THEN 4
                WHEN p.primary_specialty LIKE '%Nurse Practitioner%' OR 
                     p.primary_specialty LIKE '%Physician Assistant%' 
                    THEN 5
                WHEN a.NPI IS NULL 
                    THEN 7
                ELSE 6
            END AS PRIORITY,
            -- Number of Visits (Condition 2)
            COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
            -- Most Recent Visit (Condition 3)
            MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
        FROM mpsii_diagnosis_table a
        LEFT JOIN com_edp_prd.com_raw.kom_providers p ON a.NPI = p.NPI
        GROUP BY 
            a.PATIENT_ID, 
            a.NPI, 
            p.primary_specialty, 
            p.secondary_specialty
    ) ranked_hcps
) final_ranking
WHERE HCP_RANK = 1
ORDER BY PATIENT_ID;

## Patient HCP Summary

In [0]:
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- =============================================================================
-- Purpose: Identify key healthcare providers for MPSII patients (Elaprase GTM)
-- Windows:
--   5 yrs: 2020-08-01..2025-11-30
--   3 yrs: 2022-08-01..2025-11-30
--   2 yrs: 2023-08-01..2025-11-30
-- Ranking basis:
--   Top 5 by 3-year visit count (Dx + Tx), tie-break: last visit (DESC), NPI (ASC)
-- Outputs include 3yr ranking, but visit counts and last visit dates use 5yr window
-- New additions: Most recent Tx HCP in 2yr, Latest treatment date in 5yr
-- MODIFICATION: Last visit dates now extended beyond 2025-11-30 to capture latest activity
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ----------------------------------------------------------
-- Claims universes (5y, 3y, 2y) Dx + Tx, with NPIs
-- ----------------------------------------------------------

cohort_3_excluded_specialties as (
  select distinct npi
from (select distinct NPI, PRIMARY_SPECIALTY, SECONDARY_SPECIALTY
from com_raw.kom_providers
where PROVIDER_TYPE = 'INDIVIDUAL')
where NOT (
    -- Primary specialty is in the exclusion list (or is NULL)
    (
        PRIMARY_SPECIALTY IN (
            'Anesthesiologist Assistant',
            'Anesthesiology',
            'Dentist',
            'Dietitian, Registered',
            'Emergency Medical Technician, Basic',
            'Emergency Medicine',
            'General Acute Care Hospital',
            'Nurse Anesthetist, Certified Registered',
            'Obstetrics & Gynecology',
            'Pathology',
            'Radiology',
            'Urology'
        )
        OR PRIMARY_SPECIALTY IS NULL
    )
    -- AND secondary specialty is NOT in the exception list
    AND (
        SECONDARY_SPECIALTY NOT IN (
            -- Behavioral/Mental Healthcare
            'Child & Adolescent Psychiatry',
            'Psychiatry',
            -- Pediatric Medicine
            'Adolescent Medicine',
            'Developmental - Behavioral Pediatrics',
            'Neonatal-Perinatal Medicine',
            'Nutrition, Pediatric',
            'Oncology, Pediatrics',
            'Pediatric Cardiology',
            'Pediatric Critical Care Medicine',
            'Pediatric Dermatology',
            'Pediatric Emergency Medicine',
            'Pediatric Endocrinology',
            'Pediatric Gastroenterology',
            'Pediatric Hematology-Oncology',
            'Pediatric Infectious Diseases',
            'Pediatric Nephrology',
            'Pediatric Ophthalmology and Strabismus Specialist',
            'Pediatric Orthopaedic Surgery',
            'Pediatric Otolaryngology',
            'Pediatric Pulmonology',
            'Pediatric Radiology',
            'Pediatric Rehabilitation Medicine',
            'Pediatric Rheumatology',
            'Pediatric Surgery',
            'Pediatrics',
            -- Medical Genetics
            'Clinical Biochemical Genetics',
            'Clinical Genetics (M.D.)',
            'Clinical Molecular Genetics',
            'Ph.D. Medical Genetics',
            -- Neurology
            'Neurodevelopmental Disabilities',
            'Neurology',
            'Neurology with Special Qualifications in Child Neurology',
            'Neuroradiology'
        )
        OR SECONDARY_SPECIALTY IS NULL
    )
)
), 

all_dx_claims_5yr AS (
    select DISTINCT *
    from (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    where npi in (select * from cohort_3_excluded_specialties) or npi is NULL 
),
all_tx_claims_5yr AS (
    select DISTINCT *
    from (SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients))
    where npi in (select * from cohort_3_excluded_specialties) or npi is NULL 
),
all_claims_5yr AS (
    SELECT DISTINCT * FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT * FROM all_tx_claims_5yr
),
all_claims_3yr AS (
    select DISTINCT *
    from (SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761','E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2022-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2022-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients))
    where npi in (select DISTINCT * from cohort_3_excluded_specialties) or npi is NULL 
),

-- ----------------------------------------------------------
-- Extended Claims for Last Visit Dates (No End Date Restriction)
-- ----------------------------------------------------------
all_claims_5yr_extended AS (
    SELECT DISTINCT * FROM (
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
          AND SERVICE_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE IN ('E761','E763')
          AND TRANSACTION_STATUS = 'PAID'
          AND FILL_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT DISTINCT * FROM cohort_3_excluded_specialties) or npi is NULL
    ),
    
all_tx_claims_5yr_extended AS (
    SELECT DISTINCT * FROM (
        SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION
        SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events 
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE >= '2020-08-01'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT DISTINCT * FROM cohort_3_excluded_specialties) or npi is NULL 
),

-- ----------------------------------------------------------
-- First Dx / First Tx (5y)
-- ----------------------------------------------------------
first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),
provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty 
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),
first_dx_hcp_last_visit_extended AS (
    SELECT fdh.PATIENT_ID, 
           MAX(ac.FILL_DATE) AS first_dx_last_visit_extended
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr_extended ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID
),
first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),
first_tx_hcp_last_visit_extended AS (
    SELECT fth.PATIENT_ID,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_extended
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr_extended ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID
),
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- ----------------------------------------------------------
-- Most-seen ranking (Top 5 by 3y) with 5y counts + 5y last-visit
-- ----------------------------------------------------------
most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),
most_seen_last_visit_extended AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        MAX(ac5.FILL_DATE) AS last_visit_extended
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr_extended ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank
),

-- ----------------------------------------------------------
-- Historical First Dates (No Date Restrictions)
-- ----------------------------------------------------------
historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),
historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),
latest_claim AS (
    SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_claim_date
    FROM all_claims_5yr
    GROUP BY PATIENT_ID
),
latest_claim_extended AS (
    SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_claim_date_extended
    FROM all_claims_5yr_extended
    GROUP BY PATIENT_ID
),
-- ----------------------------------------------------------
-- Most Recent Treatment HCP (2-year window)
-- ----------------------------------------------------------
all_tx_claims_2yr AS (
    SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    UNION
    SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                             '38206','38230','38232','38240','38241','38242','38243','38250')
      AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
),
most_recent_tx_hcp_2yr_ranked AS (
    SELECT PATIENT_ID, NPI, MAX(FILL_DATE) AS most_recent_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MAX(FILL_DATE) DESC, NPI ASC) AS rn
    FROM all_tx_claims_2yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
most_recent_tx_hcp_2yr AS (
    SELECT PATIENT_ID, 
           NPI AS most_recent_tx_hcp_2yr
    FROM most_recent_tx_hcp_2yr_ranked
    WHERE rn = 1
),
most_recent_tx_hcp_2yr_5yr_stats AS (
    SELECT 
        mrtx.PATIENT_ID, 
        mrtx.most_recent_tx_hcp_2yr,
        MAX(ac5.FILL_DATE) AS most_recent_tx_hcp_2yr_last_visit_5yr
    FROM most_recent_tx_hcp_2yr mrtx
    LEFT JOIN all_claims_5yr ac5
      ON mrtx.PATIENT_ID = ac5.PATIENT_ID 
      AND mrtx.most_recent_tx_hcp_2yr = ac5.NPI
    GROUP BY mrtx.PATIENT_ID, mrtx.most_recent_tx_hcp_2yr
),
most_recent_tx_hcp_last_visit_extended AS (
    SELECT 
        mrtx.PATIENT_ID, 
        MAX(ac5.FILL_DATE) AS most_recent_tx_hcp_last_visit_extended
    FROM most_recent_tx_hcp_2yr mrtx
    LEFT JOIN all_claims_5yr_extended ac5
      ON mrtx.PATIENT_ID = ac5.PATIENT_ID 
      AND mrtx.most_recent_tx_hcp_2yr = ac5.NPI
    GROUP BY mrtx.PATIENT_ID
),
-- ----------------------------------------------------------
-- Latest Treatment Date (5-year window)
-- ----------------------------------------------------------
latest_treatment_date_5yr AS (
    SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_treatment_date_5yr
    FROM all_tx_claims_5yr
    GROUP BY PATIENT_ID
),
latest_treatment_date_extended AS (
    SELECT PATIENT_ID, MAX(FILL_DATE) AS latest_treatment_date_extended
    FROM all_tx_claims_5yr_extended
    GROUP BY PATIENT_ID
),
-- Patient Level Information
patient_demographics AS (
    SELECT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER
    FROM com_edp_prd.com_raw.kom_patient_demographics
),
patient_geography AS (
    select distinct patient_id, patient_state
    from (SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID 
                ORDER BY 
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,  -- Prioritize current records
                    VALID_TO_DATE DESC  -- Then most recent valid_to_date
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    ) ranked
    WHERE rn = 1  -- Take only the best record per patient))
)),

/* =========================================================
   Latest Claim HCP (EXTENDED, ONE-TO-ONE)
   ========================================================= */
latest_claim_hcp_extended_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_claims_5yr_extended
    WHERE NPI IS NOT NULL
),
latest_claim_hcp_extended AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_claim_hcp_npi
    FROM latest_claim_hcp_extended_ranked
    WHERE rn = 1
),
latest_claim_hcp_visit_count AS (
    SELECT
        lch.PATIENT_ID,
        lch.latest_claim_hcp_npi,
        COUNT(DISTINCT ac.FILL_DATE) AS latest_claim_hcp_visit_count
    FROM latest_claim_hcp_extended lch
    LEFT JOIN all_claims_5yr_extended ac
      ON lch.PATIENT_ID = ac.PATIENT_ID
     AND lch.latest_claim_hcp_npi = ac.NPI
    GROUP BY lch.PATIENT_ID, lch.latest_claim_hcp_npi
),

/* =========================================================
   Most Recent Treatment HCP (EXTENDED, ONE-TO-ONE)
   ========================================================= */
most_recent_tx_hcp_extended_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_tx_claims_5yr_extended
    WHERE NPI IS NOT NULL
),
most_recent_tx_hcp_extended AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_treatment_hcp_npi
    FROM most_recent_tx_hcp_extended_ranked
    WHERE rn = 1
),
latest_treatment_hcp_visit_count AS (
    SELECT
        mrt.PATIENT_ID,
        mrt.latest_treatment_hcp_npi,
        COUNT(DISTINCT tx.FILL_DATE) AS latest_treatment_hcp_visit_count
    FROM most_recent_tx_hcp_extended mrt
    LEFT JOIN all_tx_claims_5yr_extended tx
      ON mrt.PATIENT_ID = tx.PATIENT_ID
     AND mrt.latest_treatment_hcp_npi = tx.NPI
    GROUP BY mrt.PATIENT_ID, mrt.latest_treatment_hcp_npi
)


-- -----------------------------
-- Final output
-- -----------------------------
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    -- Historical First Dates
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    -- Most Recent Tx (2yr)
    mrtx2.most_recent_tx_hcp_2yr,
    mrtxlve.most_recent_tx_hcp_last_visit_extended AS most_recent_tx_hcp_2yr_last_visit_5yr,

    -- Latest Claim (extended)
    lce.latest_claim_date_extended AS latest_claim_date,

    -- 🔹 NEW: Latest Claim HCP details
    lch.latest_claim_hcp_npi,
    pd1.provider_name     AS latest_claim_hcp_name,
    pd1.primary_specialty AS latest_claim_hcp_specialty,
    lchvc.latest_claim_hcp_visit_count,

   -- 🔹 NEW: Latest Claim HCO details
    ref1.hcp_npi AS latest_claim_hcp_hco_npi,
    ref1.hcp_name AS latest_claim_hcp_hco_name,

    -- Latest Treatment Date (extended)
    ltxe.latest_treatment_date_extended AS latest_treatment_date,

    -- 🔹 NEW: Latest Treatment HCP details
    mrt.latest_treatment_hcp_npi,
    pd2.provider_name     AS latest_treatment_hcp_name,
    pd2.primary_specialty AS latest_treatment_hcp_specialty,
    lthvc.latest_treatment_hcp_visit_count,

    -- 🔹 NEW: Latest Treatment HCO details
    ref2.hcp_npi AS latest_treatment_hcp_hco_npi,
    ref2.hcp_name AS latest_treatment_hcp_hco_name,
    
    -- First Dx HCP
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhlve.first_dx_last_visit_extended AS first_dx_last_visit_5yr,

    -- First Tx HCP
    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fthlve.first_tx_last_visit_extended AS first_tx_last_visit_5yr,

    -- Most Seen HCP #1..#5 (ranked by 3-year activity, showing 5-year visit metrics)
    
    -- #1 Most Seen
    ms1.NPI  AS most_seen_hcp1_3yr_ranked,
    COALESCE(ms1.visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    mslve1.last_visit_extended AS most_seen_hcp1_last_visit_5yr,
    
    -- #2 Most Seen
    ms2.NPI  AS most_seen_hcp2_3yr_ranked,
    COALESCE(ms2.visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    mslve2.last_visit_extended AS most_seen_hcp2_last_visit_5yr,

    -- #3 Most Seen
    ms3.NPI  AS most_seen_hcp3_3yr_ranked,
    COALESCE(ms3.visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    mslve3.last_visit_extended AS most_seen_hcp3_last_visit_5yr,
    
    -- #4 Most Seen
    ms4.NPI  AS most_seen_hcp4_3yr_ranked,
    COALESCE(ms4.visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    mslve4.last_visit_extended AS most_seen_hcp4_last_visit_5yr,
    
    -- #5 Most Seen
    ms5.NPI  AS most_seen_hcp5_3yr_ranked,
    COALESCE(ms5.visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    mslve5.last_visit_extended AS most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd          ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg             ON ep.PATIENT_ID = pg.PATIENT_ID
LEFT JOIN historical_first_dx hfdx         ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx         ON ep.PATIENT_ID = hftx.PATIENT_ID
-- LEFT JOIN latest_claim lc                  ON ep.PATIENT_ID = lc.PATIENT_ID
LEFT JOIN latest_claim_extended lce        ON ep.PATIENT_ID = lce.PATIENT_ID
--Latest Claim HCP joins
LEFT JOIN latest_claim_hcp_extended lch      ON ep.PATIENT_ID = lch.PATIENT_ID
LEFT JOIN latest_claim_hcp_visit_count lchvc  ON ep.PATIENT_ID = lchvc.PATIENT_ID
LEFT JOIN provider_dim pd1                  ON lch.latest_claim_hcp_npi = pd1.npi
--Latest Treatment HCP joins
LEFT JOIN most_recent_tx_hcp_extended mrt   ON ep.PATIENT_ID = mrt.PATIENT_ID
LEFT JOIN latest_treatment_hcp_visit_count lthvc  ON ep.PATIENT_ID = lthvc.PATIENT_ID
LEFT JOIN provider_dim pd2 ON mrt.latest_treatment_hcp_npi = pd2.npi
--HCO Name join
LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 ref1 ON lch.latest_claim_hcp_npi = ref1.hcp_npi
LEFT JOIN cmpa_insights_internal_schema.reference_file_0109 ref2 ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi
LEFT JOIN most_recent_tx_hcp_2yr mrtx2     ON ep.PATIENT_ID = mrtx2.PATIENT_ID
LEFT JOIN most_recent_tx_hcp_2yr_5yr_stats mrtx2stats ON ep.PATIENT_ID = mrtx2stats.PATIENT_ID
LEFT JOIN most_recent_tx_hcp_last_visit_extended mrtxlve ON ep.PATIENT_ID = mrtxlve.PATIENT_ID
LEFT JOIN latest_treatment_date_5yr ltx5   ON ep.PATIENT_ID = ltx5.PATIENT_ID
LEFT JOIN latest_treatment_date_extended ltxe ON ep.PATIENT_ID = ltxe.PATIENT_ID
LEFT JOIN first_dx_hcp fdh                 ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs      ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_dx_hcp_last_visit_extended fdhlve ON ep.PATIENT_ID = fdhlve.PATIENT_ID
LEFT JOIN first_tx_hcp fth                 ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths      ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_last_visit_extended fthlve ON ep.PATIENT_ID = fthlve.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx   ON ep.PATIENT_ID = fthtx.PATIENT_ID
LEFT JOIN most_seen_combined_stats ms1     ON ep.PATIENT_ID = ms1.PATIENT_ID AND ms1.rank = 1
LEFT JOIN most_seen_last_visit_extended mslve1 ON ep.PATIENT_ID = mslve1.PATIENT_ID AND mslve1.rank = 1
LEFT JOIN most_seen_combined_stats ms2     ON ep.PATIENT_ID = ms2.PATIENT_ID AND ms2.rank = 2
LEFT JOIN most_seen_last_visit_extended mslve2 ON ep.PATIENT_ID = mslve2.PATIENT_ID AND mslve2.rank = 2
LEFT JOIN most_seen_combined_stats ms3     ON ep.PATIENT_ID = ms3.PATIENT_ID AND ms3.rank = 3
LEFT JOIN most_seen_last_visit_extended mslve3 ON ep.PATIENT_ID = mslve3.PATIENT_ID AND mslve3.rank = 3
LEFT JOIN most_seen_combined_stats ms4     ON ep.PATIENT_ID = ms4.PATIENT_ID AND ms4.rank = 4
LEFT JOIN most_seen_last_visit_extended mslve4 ON ep.PATIENT_ID = mslve4.PATIENT_ID AND mslve4.rank = 4
LEFT JOIN most_seen_combined_stats ms5     ON ep.PATIENT_ID = ms5.PATIENT_ID AND ms5.rank = 5
LEFT JOIN most_seen_last_visit_extended mslve5 ON ep.PATIENT_ID = mslve5.PATIENT_ID AND mslve5.rank = 5
ORDER BY ep.PATIENT_ID;

-- Create the table from the temp view
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_base

# **Purpose**

**mpsii_tx_claims** creates a list of **treatment claims** for MPS II patients (Elaprase + related procedures).

It only includes **patients with at least 2** E761 diagnosis dates and treatment between **2023-08-01 and 2025-11-30**.

# **Patient Eligibility**

A patient is included if:

- They have **≥ 2 distinct E761 diagnosis dates** (medical or pharmacy)
- Diagnoses occurred between **2020-08-01** and **2025-11-30**

This ensures patients are confirmed MPS II cases before pulling their treatment records.

# **What Counts as a Treatment Claim**

The view combines three treatment sources:

1. **Medical NDC claims**

    - NDC11 IN ('54092070001','540920700')

2. **Pharmacy NDC claims (paid only)**

    - Same NDCs
    - TRANSACTION_RESULT = 'PAID'

3. **Medical procedure-code claims**

    - Codes related to Elaprase administration (e.g., J1743, 99601, infusion codes, etc.)

**All claims must**:

- Belong to an eligible MPS II patient
- Have a treatment date between 2023-08-01 and 2025-11-30

# **Output Columns**

Each row represents one treatment event, with:

- Patient ID
- NPI (treating provider)
- Code (NDC or procedure code)
- Event ID
- Fill/Service Date
- Place of Service
- KH Plan
- Source Table (medical or pharmacy)

In [0]:
CREATE OR replace temp VIEW mpsii_tx_claims AS 
(
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
where (patient_id in (SELECT DISTINCT PATIENT_ID AS PATIENT_ID 
FROM 
(
SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS
FROM (SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '${end_date}')
GROUP BY PATIENT_ID
)
WHERE NUMBER_OF_CLAIMS >=2)) and fill_date between '2023-08-01' AND '${end_date}'
);

### Appending New Columns (Recently Treated HCP), Primary HCP in 2 year

# **Purpose**

Build a patient-level view that shows, for each MPS II patient:

- Their most recently treating HCP in the last 2 years (from mpsii_tx_claims), and

- How often this HCP has seen the patient over the last 5 years (Dx + Tx), plus

- HCP and HCO details (name, specialty, zip, territory, region).

# **Main Logic (step-by-step)**

1. **tx_claims**

    - All distinct treatment claims from mpsii_tx_claims (already 2-year Tx-filtered).

2. **all_dx_claims_5yr**

    - E761 diagnosis claims (medical + pharmacy) in 2020-08-01 to 2025-11-30,
    - Only for patients appearing in tx_claims.

3. **all_tx_claims_5yr**

    - Treatment claims (same Tx logic as mpsii_tx_claims) in the same 5-year window,
    - Only for patients appearing in tx_claims.

4. **all_claims_5yr**
    - Union of 5-year Dx + Tx → full 5-year visit universe per patient–HCP.

5. **latest_treating_hcp**

    - From tx_claims (2-year Tx):

      - For each patient, pick one HCP (NPI):

        - Prefer non-null NPI,

        - Then most recent fill_date,

        - Then highest NPI (tie-break).

    - Keeps the most recently treating HCP per patient.

6. **visit_counts**

- From all_claims_5yr:

    - Count distinct fill_date per (patient_id, npi)
    → 5-year visit_counts (Dx + Tx).

7. **latest_treating_hcp_with_visits**

- Join the most recent HCP (2-year) with 5-year visit_counts
→ how many times that HCP has seen the patient in 5 years.

8. **hcp_with_other_info**

- Add HCP details from kom_providers (INDIVIDUAL): name + primary specialty.
- Add HCO details from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109: hco_npi, hco_name, zip, territory, region.

# **Final Output Columns (per patient)**

- patient_id
- **most_recently_treated_hcp_2yr** =
NPI of most recent treating HCP (based on 2-year Tx).
- **most_recently_treated_hcp_name_2yr** =
HCP full name.
- **most_recently_treated_hcp_2yr_no_of_visits_5yr** =
Number of distinct visits (Dx + Tx) in 5 years with this HCP.
- most_recently_treated_hcp_specialty_2yr
- most_recently_treated_hcp_hco_npi
- most_recently_treated_hcp_hco_name
- most_recently_treated_hcp_territory_2yr
- most_recently_treated_hcp_region_2yr

In [0]:
select * from cmpa_insights_internal_schema.patient360_master

In [0]:
CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH 
-- 2-year treatment claims (already filtered in mpsii_tx_claims)
tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

-- 5-year Dx claims (E761) for the same patients - ORIGINAL (for reference only)
all_dx_claims_5yr AS (
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)
),

-- 5-year Tx claims - ORIGINAL (for reference only)
all_tx_claims_5yr AS (
    -- Medical events with NDC codes
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    -- Pharmacy events with NDC codes
    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    -- Medical events with procedure codes
    SELECT DISTINCT 
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)
),

-- EXTENDED Dx claims (NO UPPER BOUND)
all_dx_claims_extended AS (
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE >= '2020-08-01'  -- NO UPPER BOUND
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE >= '2020-08-01'  -- NO UPPER BOUND
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)
),

-- EXTENDED Tx claims (NO UPPER BOUND)
all_tx_claims_extended AS (
    -- Medical events with NDC codes
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE >= '2020-08-01'  -- NO UPPER BOUND
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    -- Pharmacy events with NDC codes
    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE >= '2020-08-01'  -- NO UPPER BOUND
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)

    UNION

    -- Medical events with procedure codes
    SELECT DISTINCT 
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events 
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
      AND SERVICE_DATE >= '2020-08-01'  -- NO UPPER BOUND
      AND PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM tx_claims)
),

-- EXTENDED Dx + Tx combined (NO UPPER BOUND)
all_claims_extended AS (
    SELECT * FROM all_dx_claims_extended
    UNION
    SELECT * FROM all_tx_claims_extended
),

-- Most recently treating HCP in the last 2 years (same as your original logic)
latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY 
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,  -- prioritize non-null
                    fill_date DESC,
                    npi DESC
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1 
      AND npi IS NOT NULL
),

-- Visit counts using EXTENDED claims (NO UPPER BOUND)
visit_counts AS (
    SELECT 
        patient_id, 
        npi, 
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims_extended  -- CHANGED: Now using extended claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

-- Last visit date using EXTENDED claims (NO UPPER BOUND)
last_visit_date AS (
    SELECT 
        lth.patient_id, 
        lth.npi,
        MAX(ac_ext.fill_date) AS last_visit_date_extended  -- CHANGED: Now using extended claims
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims_extended ac_ext  -- CHANGED: Now using extended claims
      ON lth.patient_id = ac_ext.patient_id 
     AND lth.npi = ac_ext.npi
    GROUP BY lth.patient_id, lth.npi
),

latest_treating_hcp_with_visits AS (
    SELECT 
        a.patient_id, 
        a.npi AS most_recently_treated_hcp, 
        b.visit_counts AS no_of_visits,
        c.last_visit_date_extended AS last_visit_date_5yr  -- CHANGED: Now using extended date
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b 
        ON a.patient_id = b.patient_id 
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id 
       AND a.npi        = c.npi
),

hcp_with_other_info AS (
    SELECT 
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        c.hco_npi,
        c.hco_name,
        c.territory_id,
        c.territory,
        c.region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a
    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b 
        ON a.most_recently_treated_hcp = b.NPI 
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'
    LEFT JOIN (SELECT
  * EXCEPT (hcp_primary_specialty),
  hcp_primary_specialty AS hcp_specialty
FROM (
  SELECT
    a.*,
    b.territory_id,
    c.region_id
  FROM cmpa_insights_internal_schema.reference_file_0109 AS a
  LEFT JOIN (
    SELECT DISTINCT
      territory_id,
      territory_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS b
    ON a.territory = b.territory_name
  LEFT JOIN (
    SELECT DISTINCT
      region_id,
      region_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS c
    ON a.region = c.region_name
)) AS c 
        ON a.most_recently_treated_hcp = c.hcp_npi  
)

SELECT 
    patient_id, 
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr, 
    hcp_name AS most_recently_treated_hcp_name_2yr, 
    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,  -- Now unbounded (all time from 2020-08-01)
    last_visit_date_5yr AS most_recently_treated_hcp_2yr_last_visit_5yr,  -- Now unbounded (all time from 2020-08-01)
    hcp_specialty AS most_recently_treated_hcp_specialty_2yr, 
    hco_npi AS most_recently_treated_hcp_hco_npi, 
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id as most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr, 
    region_id as most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
FROM hcp_with_other_info;

In [0]:
select * from most_recently_treated_hcp limit 100;

# **Primary HCP: Purpose**

This view identifies the **primary treating HCP** for each patient based on treatment activity in the last 2 years.
Selection is based on **specialty importance**, **number of visits**, and **recency of visits**, and includes HCP/HCO details.

# **How It Works**
1. **Identify each HCP’s specialty category**

    From mpsii_tx_claims, each HCP is classified into one of these categories (in order of priority):

    1. Geneticist
    2. Psychiatry & Neurology
    3. Pediatrician
    4. PCP (Internal or Family Medicine)
    5. NPPA (Nurse Practitioner / Physician Assistant)
    6. Others
    7. NA (no NPI)

    This classification comes from provider primary/secondary specialty fields.

2. **Rank HCPs per patient**

    For each patient, all treating HCPs over the last 2 years are ranked using:

    1. Specialty priority (higher priority first)

    2. Number of distinct treatment visits

    3. More recent visit dates

    4. NPI (tie-breaker)

    The HCP with rank = 1 becomes the primary HCP.

3. **Attach HCP & HCO information**

    The selected primary HCP is enriched with:

      - HCP name
      - HCP primary specialty
      - HCO NPI
      - HCO name
      - Zip code
      - Territory
      - Region

    (From provider and affiliation table - reference file.)

4. **Count 2-year visit frequency**

    Counts how many distinct treatment dates the patient had with this primary HCP.

# **Final Output Columns**

- **patient_id**
- **primary_hcp_2yr** — HCP NPI
- **primary_hcp_name_2yr**
- **primary_hcp_no_of_visits_2yr**
- **primary_hcp_specialty_2yr**
- **primary_hcp_hco_npi_2yr**
- **primary_hcp_hco_name_2yr**
- **primary_hcp_territory_2yr**
- **primary_hcp_region_2yr**

In [0]:
  -- =============================================================================
  -- Primary HCP Assignment (Dx + Tx Claims) - CORRECTED
  -- =============================================================================
  -- Diagnosis Window: 2020-08-01 to 2025-11-30 (5 years)
  -- Treatment Window: 2023-08-01 to 2025-11-30 (2 years)
  -- 
  -- Primary HCP considers BOTH Dx and Tx claims for visit counting
  -- 
  -- NPI LOGIC (aligned with GTM file):
  --   - NDC (Medical): COALESCE(RENDERING_NPI, REFERRING_NPI)
  --   - Procedure (Medical): RENDERING_NPI only
  --   - Pharmacy: PRESCRIBER_NPI
  -- =============================================================================


  -- =============================================================================
  -- STEP 1: DIAGNOSIS CLAIMS (5yr)
  -- =============================================================================

  CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

  -- Medical Events - Dx (COALESCE)
  SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      'DX' AS CLAIM_TYPE
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

  UNION

  -- Pharmacy Events - Dx (PRESCRIBER_NPI)
  SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      'DX' AS CLAIM_TYPE
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
    AND TRANSACTION_STATUS = 'PAID'
    AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


  -- =============================================================================
  -- STEP 2: TREATMENT CLAIMS (5yr) - CORRECTED NPI LOGIC
  -- =============================================================================

  CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

  -- Medical Events - NDC codes (COALESCE)
  SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      'TX' AS CLAIM_TYPE,
      NDC11 AS CODE
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE NDC11 IN ('54092070001', '540920700')
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

  UNION

  -- Medical Events - Procedure codes (RENDERING_NPI only)
  SELECT DISTINCT 
      PATIENT_ID,
      RENDERING_NPI AS NPI,
      SERVICE_DATE AS FILL_DATE,
      'TX' AS CLAIM_TYPE,
      PROCEDURE_CODE AS CODE
  FROM com_edp_prd.com_raw.kom_medical_events
  WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                          'S9357', 'S9379', '38206', '38230', '38232', 
                          '38240', '38241', '38242', '38243', '38250')
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

  UNION

  -- Pharmacy Events - NDC codes (PRESCRIBER_NPI)
  SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      'TX' AS CLAIM_TYPE,
      NDC11 AS CODE
  FROM com_edp_prd.com_raw.kom_pharmacy_events
  WHERE NDC11 IN ('54092070001', '540920700')
    AND TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';


  -- =============================================================================
  -- STEP 3: TREATMENT CLAIMS FOR ELIGIBILITY (2yr)
  -- =============================================================================

  CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
  SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
  FROM all_tx_claims
  WHERE FILL_DATE BETWEEN '2023-08-01' AND '${end_date}';


  -- =============================================================================
  -- STEP 4: PATIENT ELIGIBILITY
  -- =============================================================================

  -- Specified: 2+ E761 Dx dates (5yr)
  CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
  SELECT PATIENT_ID
  FROM (
      SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE DIAGNOSIS_CODES LIKE '%E761%'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      UNION
      SELECT DISTINCT PATIENT_ID, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE = 'E761'
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
  )
  GROUP BY PATIENT_ID
  HAVING COUNT(DISTINCT FILL_DATE) >= 2;


  -- Specified Patients: 2+ E761 Dx + any Tx in 2yr
  CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
  SELECT DISTINCT e.PATIENT_ID
  FROM e761_patients_2dx e
  INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;


  -- Incremental: 2+ E763 Dx dates (5yr)
  CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
  SELECT PATIENT_ID
  FROM (
      SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE DIAGNOSIS_CODES LIKE '%E763%'
        AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      UNION
      SELECT DISTINCT PATIENT_ID, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE = 'E763'
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
  )
  GROUP BY PATIENT_ID
  HAVING COUNT(DISTINCT FILL_DATE) >= 2;


  -- Elaprase Tx in 2yr (for incremental eligibility)
  CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
  SELECT DISTINCT PATIENT_ID
  FROM tx_claims_2yr
  WHERE CODE IN ('54092070001', '540920700', 'J1743');


  -- Incremental Patients: 2+ E763 Dx + Elaprase Tx in 2yr + NOT specified
  CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
  SELECT DISTINCT e.PATIENT_ID
  FROM e763_patients_2dx e
  INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
  WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


  -- All Eligible Patients
  CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
  SELECT PATIENT_ID FROM specified_patients
  UNION
  SELECT PATIENT_ID FROM incremental_patients;


  -- =============================================================================
  -- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
  -- =============================================================================

  CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

  -- Dx claims (5yr)
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  -- Tx claims (5yr)
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);


  -- =============================================================================
  -- STEP 6: PRIMARY HCP ASSIGNMENT WITH 4-TIER RANKING
  -- =============================================================================

  CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
  WITH hcp_metrics AS (
      SELECT 
          a.PATIENT_ID,
          a.NPI,
          
          -- Specialty Classification
          CASE 
              WHEN p.primary_specialty LIKE '%Genetic%' 
                OR p.secondary_specialty LIKE '%Genetic%' 
                  THEN 'Geneticist'
              WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
                OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
                OR p.primary_specialty LIKE '%Neurological Surgery%' 
                  THEN 'Psychiatry & Neurology'
              WHEN p.primary_specialty LIKE '%Pediatrics%' 
                  THEN 'Pediatrician'
              WHEN p.primary_specialty LIKE '%Internal Medicine%' 
                OR p.secondary_specialty LIKE '%Internal Medicine%'
                OR p.primary_specialty LIKE '%Family Medicine%' 
                OR p.secondary_specialty LIKE '%Family Medicine%' 
                  THEN 'PCP'
              WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
                OR p.primary_specialty LIKE '%Physician Assistant%' 
                  THEN 'NPPA'
              WHEN a.NPI IS NULL 
                  THEN 'NA'
              ELSE 'Others'
          END AS SPECIALTY,
          
          -- Priority (Tier 1)
          CASE 
              WHEN p.primary_specialty LIKE '%Genetic%' 
                OR p.secondary_specialty LIKE '%Genetic%' 
                  THEN 1
              WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' 
                OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' 
                OR p.primary_specialty LIKE '%Neurological Surgery%' 
                  THEN 2
              WHEN p.primary_specialty LIKE '%Pediatrics%' 
                  THEN 3
              WHEN p.primary_specialty LIKE '%Internal Medicine%' 
                OR p.secondary_specialty LIKE '%Internal Medicine%'
                OR p.primary_specialty LIKE '%Family Medicine%' 
                OR p.secondary_specialty LIKE '%Family Medicine%' 
                  THEN 4
              WHEN p.primary_specialty LIKE '%Nurse Practitioner%' 
                OR p.primary_specialty LIKE '%Physician Assistant%' 
                  THEN 5
              WHEN a.NPI IS NULL 
                  THEN 7
              ELSE 6
          END AS SPECIALTY_PRIORITY,
          
          -- Visit Count: Dx + Tx combined (Tier 2)
          COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
          
          -- Dx-only visit count (for reference)
          COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
          
          -- Tx-only visit count (for reference)
          COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,
          
          -- Most Recent Visit (Tier 3)
          MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
          
      FROM all_patient_claims a
      LEFT JOIN com_edp_prd.com_raw.kom_providers p 
          ON a.NPI = p.NPI
      GROUP BY 
          a.PATIENT_ID, 
          a.NPI,
          p.primary_specialty, 
          p.secondary_specialty
  ),
  ranked_hcps AS (
      SELECT 
          *,
          RANK() OVER (
              PARTITION BY PATIENT_ID 
              ORDER BY 
                  SPECIALTY_PRIORITY ASC,    -- Tier 1: Specialty (lower = better)
                  NO_OF_VISITS DESC,         -- Tier 2: Total visits (higher = better)
                  MOST_RECENT_VISIT DESC,    -- Tier 3: Recency (more recent = better)
                  NPI ASC                    -- Tier 4: NPI tiebreaker (lower wins)
          ) AS HCP_RANK
      FROM hcp_metrics
  )
  SELECT 
      PATIENT_ID,
      NPI AS PRIMARY_HCP_NPI,
      SPECIALTY AS PRIMARY_HCP_SPECIALTY,
      SPECIALTY_PRIORITY,
      NO_OF_VISITS,
      DX_VISITS,
      TX_VISITS,
      MOST_RECENT_VISIT,
      HCP_RANK
  FROM ranked_hcps
  WHERE HCP_RANK = 1;

  -- -- Creating a table for Primary HCP
  -- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
  -- SELECT * FROM primary_hcp_assignment;

In [0]:
-- =============================================================================
-- STEP 7: CREATE PRIMARY HCP TABLE WITH NAMES AND TERRITORY
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
SELECT 
    ph.*,
    COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,
    ref.HCO_NPI AS primary_hcp_hco_npi_2yr,
    ref.HCO_NAME AS primary_hcp_hco_name_2yr,
    ref.territory_id as primary_hcp_territory_id_2yr,
    ref.TERRITORY AS primary_hcp_territory_2yr,
    ref.region_id as primary_hcp_region_id_2yr,
    ref.region as primary_hcp_region_2yr
FROM primary_hcp ph
LEFT JOIN com_edp_prd.com_raw.kom_providers p 
    ON ph.PRIMARY_HCP_NPI = p.NPI
LEFT JOIN (SELECT
  * EXCEPT (hcp_primary_specialty),
  hcp_primary_specialty AS hcp_specialty
FROM (
  SELECT
    a.*,
    b.territory_id,
    c.region_id
  FROM cmpa_insights_internal_schema.reference_file_0109 AS a
  LEFT JOIN (
    SELECT DISTINCT
      territory_id,
      territory_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS b
    ON a.territory = b.territory_name
  LEFT JOIN (
    SELECT DISTINCT
      region_id,
      region_name
    FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) AS c
    ON a.region = c.region_name
)
) ref
    ON ph.PRIMARY_HCP_NPI = ref.HCP_NPI;

In [0]:
-- =============================================================================
-- 3. SUMMARY: CLAIMS BY NPI FOR THIS PATIENT
-- =============================================================================

WITH all_claims AS (
    -- Dx Claims (Medical)
    SELECT DISTINCT
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID = 'DV87FE07'
      AND (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Dx Claims (Pharmacy)
    SELECT DISTINCT
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID = 'DV87FE07'
      AND DIAGNOSIS_CODE IN ('E761', 'E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims - NDC (Medical)
    SELECT DISTINCT
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID = 'DV87FE07'
      AND NDC11 IN ('54092070001', '540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims - Procedure (Medical) - RENDERING_NPI only
    SELECT DISTINCT
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID = 'DV87FE07'
      AND PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                             'S9357', 'S9379', '38206', '38230', '38232', 
                             '38240', '38241', '38242', '38243', '38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims (Pharmacy)
    SELECT DISTINCT
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID = 'DV87FE07'
      AND NDC11 IN ('54092070001', '540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
)

SELECT 
    NPI,
    COUNT(DISTINCT FILL_DATE) AS TOTAL_VISITS,
    COUNT(DISTINCT CASE WHEN CLAIM_TYPE = 'DX' THEN FILL_DATE END) AS DX_VISITS,
    COUNT(DISTINCT CASE WHEN CLAIM_TYPE = 'TX' THEN FILL_DATE END) AS TX_VISITS,
    MIN(FILL_DATE) AS FIRST_VISIT,
    MAX(FILL_DATE) AS MOST_RECENT_VISIT
FROM all_claims
GROUP BY NPI
ORDER BY TOTAL_VISITS DESC;

In [0]:
-- =============================================================================
-- Primary HCP Rationale for Selected Patients
-- =============================================================================

WITH all_claims AS (
    -- Dx Claims (Medical)
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN ('0QMXBD93','6XZ63TZJ','BTEQP2RC','GESKBZTW','SNHEKHZZ',
                         '7M0D8HCK','CPV0FW0X','KP8TFZE3','M1GQXFYB','NFC557NB',
                         'L4PXNEW9','MBCKCXPB','R6BP8FQ1','R880T4GS','T84CH62W',
                         'VM0F6LD8','Y416J3WM')
      AND (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Dx Claims (Pharmacy)
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID IN ('0QMXBD93','6XZ63TZJ','BTEQP2RC','GESKBZTW','SNHEKHZZ',
                         '7M0D8HCK','CPV0FW0X','KP8TFZE3','M1GQXFYB','NFC557NB',
                         'L4PXNEW9','MBCKCXPB','R6BP8FQ1','R880T4GS','T84CH62W',
                         'VM0F6LD8','Y416J3WM')
      AND DIAGNOSIS_CODE IN ('E761', 'E763')
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims - NDC (Medical) - COALESCE
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN ('0QMXBD93','6XZ63TZJ','BTEQP2RC','GESKBZTW','SNHEKHZZ',
                         '7M0D8HCK','CPV0FW0X','KP8TFZE3','M1GQXFYB','NFC557NB',
                         'L4PXNEW9','MBCKCXPB','R6BP8FQ1','R880T4GS','T84CH62W',
                         'VM0F6LD8','Y416J3WM')
      AND NDC11 IN ('54092070001', '540920700')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims - Procedure (Medical) - RENDERING_NPI only
    SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID IN ('0QMXBD93','6XZ63TZJ','BTEQP2RC','GESKBZTW','SNHEKHZZ',
                         '7M0D8HCK','CPV0FW0X','KP8TFZE3','M1GQXFYB','NFC557NB',
                         'L4PXNEW9','MBCKCXPB','R6BP8FQ1','R880T4GS','T84CH62W',
                         'VM0F6LD8','Y416J3WM')
      AND PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743', 
                             'S9357', 'S9379', '38206', '38230', '38232', 
                             '38240', '38241', '38242', '38243', '38250')
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'

    UNION

    -- Tx Claims (Pharmacy)
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID IN ('0QMXBD93','6XZ63TZJ','BTEQP2RC','GESKBZTW','SNHEKHZZ',
                         '7M0D8HCK','CPV0FW0X','KP8TFZE3','M1GQXFYB','NFC557NB',
                         'L4PXNEW9','MBCKCXPB','R6BP8FQ1','R880T4GS','T84CH62W',
                         'VM0F6LD8','Y416J3WM')
      AND NDC11 IN ('54092070001', '540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),

hcp_metrics AS (
    SELECT 
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty,
        
        -- Specialty Classification
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' OR p.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR p.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%' OR p.secondary_specialty LIKE '%Internal Medicine%' OR p.primary_specialty LIKE '%Family Medicine%' OR p.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' OR p.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
            WHEN a.NPI IS NULL THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,
        
        -- Priority (Tier 1)
        CASE 
            WHEN p.primary_specialty LIKE '%Genetic%' OR p.secondary_specialty LIKE '%Genetic%' THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%' OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%' OR p.primary_specialty LIKE '%Neurological Surgery%' THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%' OR p.secondary_specialty LIKE '%Internal Medicine%' OR p.primary_specialty LIKE '%Family Medicine%' OR p.secondary_specialty LIKE '%Family Medicine%' THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%' OR p.primary_specialty LIKE '%Physician Assistant%' THEN 5
            WHEN a.NPI IS NULL THEN 7
            ELSE 6
        END AS PRIORITY,
        
        -- Metrics
        COUNT(DISTINCT a.FILL_DATE) AS TOTAL_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,
        MIN(a.FILL_DATE) AS FIRST_VISIT,
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
        
    FROM all_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p ON a.NPI = p.NPI
    GROUP BY 
        a.PATIENT_ID,
        a.NPI, 
        p.primary_specialty, 
        p.secondary_specialty
),

ranked_hcps AS (
    SELECT 
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID 
            ORDER BY 
                PRIORITY ASC,
                TOTAL_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)

SELECT 
    PATIENT_ID,
    NPI,
    SPECIALTY,
    PRIORITY,
    TOTAL_VISITS,
    DX_VISITS,
    TX_VISITS,
    FIRST_VISIT,
    MOST_RECENT_VISIT,
    HCP_RANK,
    CASE 
        WHEN HCP_RANK = 1 THEN 'PRIMARY - Selected'
        ELSE 'Not Selected'
    END AS STATUS,
    CASE 
        WHEN HCP_RANK = 1 THEN
            CASE 
                WHEN PRIORITY = (SELECT MIN(PRIORITY) FROM ranked_hcps r2 WHERE r2.PATIENT_ID = ranked_hcps.PATIENT_ID) 
                     AND (SELECT COUNT(DISTINCT NPI) FROM ranked_hcps r3 WHERE r3.PATIENT_ID = ranked_hcps.PATIENT_ID AND r3.PRIORITY = ranked_hcps.PRIORITY) = 1
                THEN 'Won by Specialty (only ' || SPECIALTY || ' for this patient)'
                WHEN PRIORITY = (SELECT MIN(PRIORITY) FROM ranked_hcps r2 WHERE r2.PATIENT_ID = ranked_hcps.PATIENT_ID)
                     AND TOTAL_VISITS = (SELECT MAX(TOTAL_VISITS) FROM ranked_hcps r3 WHERE r3.PATIENT_ID = ranked_hcps.PATIENT_ID AND r3.PRIORITY = ranked_hcps.PRIORITY)
                     AND (SELECT COUNT(DISTINCT NPI) FROM ranked_hcps r4 WHERE r4.PATIENT_ID = ranked_hcps.PATIENT_ID AND r4.PRIORITY = ranked_hcps.PRIORITY AND r4.TOTAL_VISITS = ranked_hcps.TOTAL_VISITS) = 1
                THEN 'Won by Visits (' || TOTAL_VISITS || ' visits, best among ' || SPECIALTY || ')'
                WHEN PRIORITY = (SELECT MIN(PRIORITY) FROM ranked_hcps r2 WHERE r2.PATIENT_ID = ranked_hcps.PATIENT_ID)
                     AND TOTAL_VISITS = (SELECT MAX(TOTAL_VISITS) FROM ranked_hcps r3 WHERE r3.PATIENT_ID = ranked_hcps.PATIENT_ID AND r3.PRIORITY = ranked_hcps.PRIORITY)
                THEN 'Won by Recency (' || MOST_RECENT_VISIT || ', tied on specialty & visits)'
                ELSE 'Won by NPI tiebreaker'
            END
        ELSE NULL
    END AS REASON_FOR_SELECTION,
    primary_specialty,
    secondary_specialty
FROM ranked_hcps
ORDER BY PATIENT_ID, HCP_RANK;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
SELECT DISTINCT
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,
    a.incidence_date,
    a.first_incidence_treatment_date,

    -- Latest Claim Date 
    a.latest_claim_date,

    -- 🔹 NEW: Latest Claim HCP details
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    -- 🔹 NEW: Latest Claim HCO details
    a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- Latest Treatment Date 
    a.latest_treatment_date,

    -- 🔹 NEW: Latest Treatment HCP details
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,

-- 🔹 NEW: Latest Treatment HCO details
    a.latest_treatment_hcp_hco_npi,
	a.latest_treatment_hcp_hco_name,

    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,
    -- Most recent Tx HCP 2 Years
    a.most_recent_tx_hcp_2yr,
    a.most_recent_tx_hcp_2yr_last_visit_5yr,
    -- Latest Treatment Date 5 Years
    b.* EXCEPT (patient_id),
    c.* EXCEPT (patient_id)
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a
LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id;

## Addition of Severity

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
/* =====================================================================
   PURPOSE
   ---------------------------------------------------------------------
   This script derives a PATIENT-LEVEL SEVERITY classification by:
   1. Looking back at diagnosis history over a fixed 5-year window
   2. Flagging encounters containing predefined SEVERITY diagnosis codes
   3. Counting distinct diagnosis dates with severity evidence
   4. Classifying patients as:
      - Severe      → 2+ distinct severity diagnosis dates
      - Attenuated  → otherwise
   ===================================================================== */


/* ---------------------------------------------------------------------
   STEP 0: Build patient-diagnosis base table
   ---------------------------------------------------------------------
   Logic:
   - Start from existing patient360 universe
   - Join medical claims to pull diagnosis history
   - Restrict to diagnosis lookback window
     (Aug 1, 2020 – Jul 31, 2025)
   --------------------------------------------------------------------- */
WITH base AS (
  SELECT DISTINCT
    a.PATIENT_ID,
    b.SERVICE_DATE,
    b.DIAGNOSIS_CODES
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360 a
  LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
    ON a.PATIENT_ID = b.PATIENT_ID
   AND b.SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
),


/* ---------------------------------------------------------------------
   STEP 1: Flag diagnosis records with SEVERITY codes
   ---------------------------------------------------------------------
   Business rule:
   - Diagnosis codes are pipe-delimited (|CODE|)
   - A record is flagged if it contains ANY predefined severity code
   - Codes include:
     • Neurological conditions
     • Developmental disorders
     • Intellectual disability
     • Congenital abnormalities
     • Developmental delay indicators
   --------------------------------------------------------------------- */
flagged AS (
  SELECT
    PATIENT_ID,
    SERVICE_DATE,
    CASE
      WHEN DIAGNOSIS_CODES IS NOT NULL AND (
           DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
        OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
        OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
        OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
        OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
        OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
        OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
        OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
        OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
        OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
        OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
        OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
        OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
      )
      THEN 1 ELSE 0
    END AS has_severity_code
  FROM base
),


/* ---------------------------------------------------------------------
   STEP 2: Aggregate severity evidence at patient level
   ---------------------------------------------------------------------
   Metrics derived:
   - count_fill_date:
       Total distinct diagnosis dates
   - severity_dx_distinct_dates:
       Distinct dates with severity codes
   - severity classification:
       • Severe     → 2+ severity diagnosis dates
       • Attenuated → <2 severity diagnosis dates
   --------------------------------------------------------------------- */
patient_with_severity_outcome AS (
  SELECT
    PATIENT_ID,
    COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,
    COUNT(DISTINCT CASE
                     WHEN has_severity_code = 1
                     THEN SERVICE_DATE
                   END) AS severity_dx_distinct_dates,
    CASE
      WHEN COUNT(DISTINCT CASE
                            WHEN has_severity_code = 1
                            THEN SERVICE_DATE
                          END) >= 2
      THEN 'Severe'
      ELSE 'Attenuated'
    END AS severity
  FROM flagged
  GROUP BY PATIENT_ID
  ORDER BY severity_dx_distinct_dates DESC, PATIENT_ID
)


/* ---------------------------------------------------------------------
   FINAL OUTPUT
   ---------------------------------------------------------------------
   - Join derived severity back to patient360
   - Preserve full patient360 record
   - Add single, clean SEVERITY attribute
   --------------------------------------------------------------------- */
SELECT
  a.*,
  b.severity
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master AS a
LEFT JOIN patient_with_severity_outcome AS b
  ON a.PATIENT_ID = b.patient_id;


## Addition of Comorbidity

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
WITH base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

-- 1) Filter events + explode diagnosis codes into one code per row
exploded_codes AS (
    SELECT
        m.patient_id,
        code
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN base_patients bp
        ON m.patient_id = bp.patient_id
    LATERAL VIEW explode(split(m.DIAGNOSIS_CODES, '\\|')) s AS code
    WHERE m.SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
      AND m.DIAGNOSIS_CODES IS NOT NULL
),

-- 2) Map each individual code to a comorbidity
comorbidity_raw AS (
    SELECT
        patient_id,
        CASE
            --------------------------------------------------------------------
            -- Respiratory issues
            --------------------------------------------------------------------
            WHEN code IN (
                'J45909','J4520','J4530','J4540','J4541','J4531','J45901','J45998',
                'J4521','J4542','J45990','J4550','J45902','J4551','J4532','J45991'
            ) THEN 'Asthma'

            WHEN code IN ('G4733','G4730','G479','G4731','G4739')
                THEN 'Sleep Apnea'

            WHEN code IN (
                'J988','J300','J301','J302','J305','J3081','J3089','J309','J310',
                'J311','J312','J320','J321','J322','J323','J324','J328','J329',
                'J330','J331','J338','J339','J340','J341','J342','J343','J3481',
                'J348200','J348201','J348202','J348210','J348211','J348212',
                'J34829','J3489','J349','J3501','J3502','J3503','J351','J352',
                'J353','J358','J359','J36','J370','J371','J3800','J3801','J3802',
                'J381','J382','J383','J384','J385','J386','J387','J390','J391',
                'J392','J393','J398','J399'
            ) THEN 'Other diseases of upper respiratory tract'

            WHEN code IN (
                'G4700','G4734','G4761','G478','G4710','G4701','G4736','G4720',
                'G4709','G4719','G4721','G4729','G4723','G47'
            ) THEN 'Sleep related disorders'

            WHEN code IN (
                'J00','J0100','J0101','J0110','J0111','J0120','J0121','J0130',
                'J0131','J0140','J0141','J0180','J0181','J0190','J0191','J020',
                'J028','J029','J0300','J0301','J0380','J0381','J0390','J0391',
                'J040','J0410','J0411','J042','J0430','J0431','J050','J0510',
                'J0511','J060','J069'
            ) THEN 'Acute upper respiratory infections'

            --------------------------------------------------------------------
            -- Ear-related Disorders
            --------------------------------------------------------------------
            WHEN code IN (
                'H6000','H6001','H6002','H6003','H6010','H6011','H6012','H6013',
                'H6020','H6021','H6022','H60311','H60312','H60319','H60321',
                'H60322','H60329','H60391','H60392','H60399','H6040','H6041',
                'H6042','H6043','H60501','H60502','H60509','H60511','H60512',
                'H60519','H60551','H60552','H60559','H60591','H60592','H60599',
                'H6060','H6061','H6062','H608X1','H608X2','H608X9','H6090',
                'H6091','H6092','H61001','H61002','H61003','H61009','H61011',
                'H61012','H61013','H61019','H61021','H61022','H61023','H61029',
                'H61031','H61032','H61033','H61039','H61321','H61322','H61323',
                'H61329','H6240','H6241','H6242','H6500','H6501','H6502','H6504',
                'H6505','H6507','H65111','H65112','H65114','H65115','H65117',
                'H65119','H65191','H65192','H65194','H65195','H65197','H65199',
                'H6520','H6521','H6522','H6530','H6531','H6532','H65411',
                'H65412','H65419','H65491','H65492','H65499','H6590','H6591',
                'H6592','H66001','H66002','H66003','H66004','H66005','H66006',
                'H66007','H66009','H66011','H66012','H66013','H66014','H66015',
                'H66016','H66017','H66019','H6611','H6612','H6620','H6621',
                'H6622','H663X1','H663X2','H663X9','H6640','H6641','H6642',
                'H6690','H6691','H6692','H671','H672','H679','H68001','H68002',
                'H68009','H68011','H68012','H68019','H68021','H68022','H68029',
                'H70001','H70002','H70009','H70011','H70012','H70019','H70091',
                'H70092','H70099','H7010','H7011','H7012','H70201','H70202',
                'H70209','H70211','H70212','H70219','H70221','H70222','H70229',
                'H70811','H70812','H70819','H70891','H70892','H70899','H7090',
                'H7091','H7092','H7100','H7101','H7102','H7110','H7111','H7112',
                'H7120','H7121','H7122','H7130','H7131','H7132','H7190','H7191',
                'H7192','H73001','H73002','H73009','H73011','H73012','H73019',
                'H73091','H73092','H73099','H7310','H7311','H7312','H7320',
                'H7321','H7322','H7411','H7412','H7413','H7419','H7440','H7441',
                'H7442','H7443','H748X1','H748X2','H748X3','H748X9','H7490',
                'H7491','H7492','H7493','H8120','H8121','H8122','H8301','H8302',
                'H8309','H9210','H9211','H9212','H9500','H9501','H9502','H9503'
            ) THEN 'Ear Infections'

            WHEN code IN (
                'H900','H9011','H9012','H902','H903','H9041','H9042','H905',
                'H906','H9071','H9072','H908','H90A11','H90A12','H90A21',
                'H90A22','H90A31','H90A32','H9101','H9102','H9103','H9109',
                'H9120','H9121','H9122','H9123','H918X1','H918X2','H918X3',
                'H918X9','H9190','H9191','H9192','H9193','P096'
            ) THEN 'Hearing loss'

            --------------------------------------------------------------------
            -- Gastrointestinal Disorders
            --------------------------------------------------------------------
            WHEN code IN (
                'K5900','K5909','K5901','K5904','K5903','K5902','K590','K5939'
            ) THEN 'Constipation'

            WHEN code IN ('R197','K591','K580','K529')
                THEN 'Diarrhea'

            WHEN code IN (
                'K4000','K4001','K4010','K4011','K4020','K4021','K4030','K4031',
                'K4040','K4041','K4090','K4091','K450','K451','K458','K460',
                'K461','K469'
            ) THEN 'Abdominal/inguinal hernia'

            --------------------------------------------------------------------
            -- Mobility Issues
            --------------------------------------------------------------------
            WHEN code IN ('Q751','Q754','Q755')
                THEN 'Dysostosis Complex'

            WHEN code IN (
                'M2560','M25611','M25612','M25619','M25621','M25622','M25629',
                'M25631','M25632','M25639','M25641','M25642','M25649','M25651',
                'M25652','M25659','M25661','M25662','M25669','M25671','M25672',
                'M25673','M25674','M25675','M25676','M2569'
            ) THEN 'Joint Stiffness'

            WHEN code IN ('G5600','G5601','G5602','G5603')
                THEN 'Carpal tunnel syndrome'

            --------------------------------------------------------------------
            -- Neurological / Developmental
            --------------------------------------------------------------------
            WHEN code IN (
                'F05','F060','F061','F062','F0630','F0631','F0632','F0633',
                'F0634','F064','F0670','F0671','F068','F070','F0781','F0789',
                'F079','F09','F22','F23','F24','F28','F29','F3010','F3011',
                'F3012','F3013','F302','F303','F304','F308','F309','F320',
                'F321','F322','F323','F324','F325','F328','F3289','F329','F32A',
                'F330','F331','F332','F333','F3340','F3341','F3342','F338',
                'F339','F340','F348','F3481','F3489','F349','F39','F410','F411',
                'F413','F418','F419','F430','F4310','F4311','F4312','F4320',
                'F4321','F4322','F4323','F4324','F4325','F4329','F438','F4389',
                'F439','F441','F442','F450','F451','F4522','F4541','F4542',
                'F54','F59','F600','F602','F603','F604','F605','F606','F6089',
                'F609','F6381','F6389','F639','F70','F71','F72','F73','F78',
                'F78A1','F78A9','F79','F800','F801','F802','F804','F8081',
                'F8082','F8089','F809','F810','F812','F8181','F8189','F819',
                'F82','F840','F843','F845','F848','F849','F88','F89','F900',
                'F901','F902','F908','F909','F910','F911','F912','F913','F918',
                'F919','F930','F938','F939','F940','F941','F942','F948','F949',
                'F950','F951','F9821','F9829','F983','F984','F985','F988',
                'F989','F99'
            ) THEN 'Behavioral Issues'

            WHEN code IN (
                'G910','G911','G912','G913','G914','G918','G919','Q038','Q039',
                'Q050','Q051','Q052','Q053','Q054','Q055','Q056','Q057','Q058',
                'Q0700','Q0702','Q0703','F445','G40001','G40009','G40011',
                'G40019','G40101','G40109','G40111','G40119','G40201','G40209',
                'G40211','G40219','G40501','G40509','G4089','R561'
            ) THEN 'CNS Issues'

            WHEN code IN ('R6250','R620','R6252','R6251','R6259','R627','R62')
                THEN 'Lack of Physiological Development'

            --------------------------------------------------------------------
            -- Chronic conditions
            --------------------------------------------------------------------
            WHEN code IN (
                'I10','I110','I129','I130','I119','I159','I160','I158','I120',
                'I161','I1310','I150'
            ) THEN 'Hypertension'

            WHEN code IN (
                'I050','I051','I052','I058','I059','I060','I061','I062','I068',
                'I069','I070','I071','I072','I078','I079','I080','I081','I082',
                'I083','I088','I089','I340','I341','I342','I348','I3481',
                'I3489','I349','I350','I351','I352','I358','I359','I360',
                'I361','I362','I368','I369','I370','I371','I372','I378','I379'
            ) THEN 'Valvular Heart Disease'

        END AS comorbidity_name
    FROM exploded_codes
    WHERE code IS NOT NULL AND code <> ''
),

-- 3) Map comorbidity -> category
comorbidity_with_category AS (
    SELECT
        patient_id,
        comorbidity_name,
        CASE
            WHEN comorbidity_name IN (
                'Asthma','Sleep Apnea','Other diseases of upper respiratory tract',
                'Sleep related disorders','Acute upper respiratory infections'
            ) THEN 'Respiratory issues'

            WHEN comorbidity_name IN ('Ear Infections','Hearing loss')
                THEN 'Ear-related disorders'

            WHEN comorbidity_name IN ('Constipation','Diarrhea','Abdominal/inguinal hernia')
                THEN 'Gastrointestinal disorders'

            WHEN comorbidity_name IN ('Dysostosis Complex','Joint Stiffness','Carpal tunnel syndrome')
                THEN 'Mobility issues'

            WHEN comorbidity_name IN ('Behavioral Issues','CNS Issues','Lack of Physiological Development')
                THEN 'Neurological disorders'

            WHEN comorbidity_name IN ('Hypertension','Valvular Heart Disease')
                THEN 'Other chronic conditions'
        END AS comorbidity_category
    FROM comorbidity_raw
    WHERE comorbidity_name IS NOT NULL
),

-- 4) Final aggregation per patient
patient_with_comorbidity_outcome as (
    SELECT
    patient_id,

    array_join(
        array_sort(collect_set(comorbidity_category)),
        ', '
    ) AS comorbidity_categories,

    size(collect_set(comorbidity_category)) AS count_of_comorbidity_categories,

    array_join(
        array_sort(collect_set(comorbidity_name)),
        ', '
    ) AS distinct_comorbidities,

    size(collect_set(comorbidity_name)) AS count_of_distinct_comorbidities

FROM comorbidity_with_category
GROUP BY patient_id
ORDER BY patient_id
)
select a.*, b.comorbidity_categories, b.count_of_comorbidity_categories, b.distinct_comorbidities, b.count_of_distinct_comorbidities
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
left join patient_with_comorbidity_outcome as b on a.patient_id=b.patient_id


## Addition of Secondary Specialty of Primary HCP

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
  select distinct a.*, b.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a 
left join com_edp_prd.com_raw.kom_providers as b on a.PRIMARY_HCP_NPI = b.npi and b.provider_type = 'INDIVIDUAL'  

**Metrics Added / Updated**

- Age Bucket - Categorized patient age groups
- Patient Zip (Zip3) - Derived from most relevant geography record
- First Treatment After Diagnosis Date
- Latest MPS II Treatment Date
- Latest MPS II Treatment Type (Elaprase vs Other ERT)
- Time from Diagnosis to First Treatment (Months)
- Treatment Duration (Months)
- Primary Payer
- Secondary Payer
- Active Insurance Group
- Elaprase Fill Count (within analysis window)

**Business Rules Applied**

- Age buckets derived from patient age at record level
- Patient geography selected using:
  - Current valid record first
  - Most recent historical record as fallback
- MPS II treatments identified using:
  - Elaprase NDCs
  - Relevant infusion and ERT procedure codes
- First treatment date must occur on or after diagnosis date
- Latest treatment determined by most recent fill/service date
- Treatment type classified as Elaprase if NDC or J-code present
- Primary and secondary payers determined by highest claim volume
- Active insurance group derived from most recent claim
- Elaprase fills counted as distinct treatment dates
- Analysis window applied where relevant: Aug 1, 2023 - Jul 31, 2025

In [0]:
/* =====================================================================
   PATIENT 360 TABLE REFRESH
   ---------------------------------------------------------------------
   Purpose:
   - Enrich the Patient 360 table with age buckets, geography, treatment
     timelines, payer information, insurance group, and Elaprase metrics
   - This script recreates the Patient 360 table with derived attributes
     using Komodo medical, pharmacy, and payer data
   ===================================================================== */

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

/* ---------------------------------------------------------------------
   Base Patient 360 table
   ---------------------------------------------------------------------
   Serves as the anchor for all patient-level enrichments
   --------------------------------------------------------------------- */
WITH base_table AS (
  SELECT DISTINCT *
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

/* ---------------------------------------------------------------------
   Age Buckets
   ---------------------------------------------------------------------
   Categorizes patients into clinically relevant age groups
   --------------------------------------------------------------------- */
age_buckets AS (
  SELECT DISTINCT
    patient_id,
    CASE
      WHEN patient_age < 5 THEN '<5 years'
      WHEN patient_age BETWEEN 5 AND 10 THEN '5 - 10 years'
      WHEN patient_age BETWEEN 11 AND 18 THEN '11 - 18 years'
      ELSE '>18 years'
    END AS age_bucket
  FROM base_table
),

/* ---------------------------------------------------------------------
   Best Patient Geography Record
   ---------------------------------------------------------------------
   Selects the most relevant geography record per patient:
   - Prioritizes current records
   - Falls back to most recently valid record
   --------------------------------------------------------------------- */
patient_geography AS (
  SELECT DISTINCT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),

/* ---------------------------------------------------------------------
   Patient Zip (Zip3)
   ---------------------------------------------------------------------
   Pulls patient-level zip from best geography record
   --------------------------------------------------------------------- */
patient_zip AS (
  SELECT DISTINCT
    a.patient_id,
    b.patient_zip AS zip3
  FROM base_table a
  LEFT JOIN patient_geography b
    ON a.patient_id = b.patient_id
),

/* ---------------------------------------------------------------------
   First Treatment After Diagnosis
   ---------------------------------------------------------------------
   Identifies the earliest MPS II treatment date occurring
   after the patient's diagnosis date
   --------------------------------------------------------------------- */
first_tx_after_diagnosis AS (
  SELECT DISTINCT
    patient_id,
    MIN(fill_date) AS first_tx_after_diagnosis
  FROM (
    SELECT DISTINCT patient_id, fill_date
    FROM (
      SELECT patient_id, service_date AS fill_date
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE ndc11 IN ('54092070001','540920700')

      UNION ALL
      SELECT DISTINCT patient_id, fill_date
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE ndc11 IN ('54092070001','540920700')
        AND transaction_result = 'PAID'

      UNION ALL
      SELECT DISTINCT patient_id, service_date AS fill_date
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE procedure_code IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
      )
    ) tx
    WHERE patient_id IN (SELECT patient_id FROM base_table)
      AND fill_date >= (
        SELECT MIN(incidence_date)
        FROM base_table b
        WHERE b.patient_id = tx.patient_id
      )
  )
  GROUP BY patient_id
),

/* ---------------------------------------------------------------------
   Latest MPS II Treatment Date
   ---------------------------------------------------------------------
   Captures the most recent MPS II treatment for each patient
   --------------------------------------------------------------------- */
latest_tx_date AS (
  SELECT DISTINCT
    patient_id,
    MAX(fill_date) AS latest_mpsii_tx_date
  FROM (
    SELECT DISTINCT patient_id, fill_date
    FROM (
      SELECT patient_id, service_date AS fill_date
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE ndc11 IN ('54092070001','540920700')

      UNION ALL
      SELECT patient_id, fill_date
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE ndc11 IN ('54092070001','540920700')
        AND transaction_result = 'PAID'

      UNION ALL
      SELECT patient_id, service_date AS fill_date
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE procedure_code IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
      )
    )
    WHERE patient_id IN (SELECT patient_id FROM base_table)
  )
  GROUP BY patient_id
),

/* ---------------------------------------------------------------------
   Latest MPS II Treatment Type
   ---------------------------------------------------------------------
   Classifies the most recent treatment as Elaprase vs other ERT
   --------------------------------------------------------------------- */
last_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN code IN ('54092070001','540920700','J1743')
        THEN 'Elaprase'
      ELSE 'Other ERT Proc'
    END AS last_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC
      ) AS rn
    FROM (
      SELECT patient_id, service_date AS fill_date, ndc11 AS code
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE ndc11 IN ('54092070001','540920700')

      UNION ALL
      SELECT patient_id, fill_date, ndc11 AS code
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE ndc11 IN ('54092070001','540920700')
        AND transaction_result = 'PAID'

      UNION ALL
      SELECT patient_id, service_date AS fill_date, procedure_code AS code
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE procedure_code IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
      )
    )
    WHERE code IS NOT NULL
  )
  WHERE rn = 1
),

/* ---------------------------------------------------------------------
   Claims-Level Payer Mapping
   ---------------------------------------------------------------------
   Joins treatment claims to payer and insurance group
   --------------------------------------------------------------------- */
tx_claims_payer_analysis AS (
  SELECT DISTINCT
    a.patient_id,
    a.fill_date,
    a.claim_id,
    a.kh_plan_id,
    b.payer_name,
    b.insurance_group
  FROM (
    SELECT patient_id, service_date AS fill_date, kh_plan_id,
           medical_event_id AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL
    SELECT patient_id, fill_date,
           COALESCE(primary_kh_plan_id, secondary_kh_plan_id),
           pharmacy_event_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE transaction_result = 'PAID'
  ) a
  LEFT JOIN com_edp_prd.com_raw.kom_plans b
    ON a.kh_plan_id = b.kh_plan_id
),

/* ---------------------------------------------------------------------
   Primary & Secondary Payer Assignment
   ---------------------------------------------------------------------
   Determined by highest claim volume within analysis window
   --------------------------------------------------------------------- */
payer_claims_count AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY patient_id
      ORDER BY num_claims DESC
    ) AS rn
  FROM (
    SELECT
      patient_id,
      payer_name,
      COUNT(DISTINCT claim_id) AS num_claims
    FROM tx_claims_payer_analysis
    WHERE payer_name IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND '${end_date}'
    GROUP BY patient_id, payer_name
  )
),

patient_primary_secondary_payer AS (
  SELECT
    patient_id,
    MAX(CASE WHEN rn = 1 THEN payer_name END) AS primary_payer,
    MAX(CASE WHEN rn = 2 THEN payer_name END) AS secondary_payer
  FROM payer_claims_count
  GROUP BY patient_id
),

/* ---------------------------------------------------------------------
   Active Insurance Group
   ---------------------------------------------------------------------
   Uses most recent insurance group from claims data
   --------------------------------------------------------------------- */
patient_active_insurance_group AS (
  SELECT
    patient_id,
    insurance_group AS active_insurance_group
  FROM (
    SELECT
      patient_id,
      insurance_group,
      fill_date,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC
      ) AS rn
    FROM tx_claims_payer_analysis
    WHERE insurance_group IS NOT NULL
  )
  WHERE rn = 1
),

/* ---------------------------------------------------------------------
   Elaprase Fill Count
   ---------------------------------------------------------------------
   Counts distinct treatment dates in the analysis window
   --------------------------------------------------------------------- */
elaprase_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS elaprase_fills
  FROM tx_claims_payer_analysis
  WHERE fill_date BETWEEN '2023-08-01' AND '${end_date}'
  GROUP BY patient_id
)

/* ---------------------------------------------------------------------
   Final Patient 360 Output
   ---------------------------------------------------------------------
   Combines all derived attributes into a single patient-level table
   --------------------------------------------------------------------- */
SELECT
  a.*,
  b.age_bucket,
  c.zip3,
  d.first_tx_after_diagnosis,
  e.latest_mpsii_tx_date,
  i.last_mpsii_tx_type AS latest_mpsii_tx_type,
  ROUND(MONTHS_BETWEEN(d.first_tx_after_diagnosis, a.incidence_date), 0)
    AS time_dx_to_first_tx_in_months,
  ROUND(MONTHS_BETWEEN(e.latest_mpsii_tx_date, d.first_tx_after_diagnosis), 0)
    AS treatment_period_months,
  f.primary_payer,
  f.secondary_payer,
  g.active_insurance_group,
  h.elaprase_fills
FROM base_table a
LEFT JOIN age_buckets b ON a.patient_id = b.patient_id
LEFT JOIN patient_zip c ON a.patient_id = c.patient_id
LEFT JOIN first_tx_after_diagnosis d ON a.patient_id = d.patient_id
LEFT JOIN latest_tx_date e ON a.patient_id = e.patient_id
LEFT JOIN patient_primary_secondary_payer f ON a.patient_id = f.patient_id
LEFT JOIN patient_active_insurance_group g ON a.patient_id = g.patient_id
LEFT JOIN elaprase_fills h ON a.patient_id = h.patient_id
LEFT JOIN last_mpsii_treatment_type i ON a.patient_id = i.patient_id;


# Adding specialties (Most recently treated hcp, primary hcp)
## We are removing old specialties column and fetching new one from komodo

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
select a.*except(a.most_recently_treated_hcp_specialty_2yr, a.PRIMARY_HCP_SPECIALTY, a.primary_hcp_secondary_specialty),
b.PRIMARY_SPECIALTY as most_recently_treated_hcp_specialty_2yr, b.SECONDARY_SPECIALTY as most_recently_treated_hcp_secondary_specialty_2yr, c.PRIMARY_SPECIALTY as primary_hcp_specialty, c.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from cmpa_insights_internal_schema.patient360_master as a
left join com_raw.kom_providers as b on a.most_recently_treated_hcp_2yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
left join com_raw.kom_providers as c on a.PRIMARY_HCP_NPI = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
select * from cmpa_insights_internal_schema.patient360_master

### Adding Newborn Screening Flag

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
with base_table as (
  select distinct * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
newborn_screening_flag as (
  select *,
  case when patient_state in ('IL', 'MO', 'WV', 'PA', 'KY', 'MD', 'CA', 'DE', 'FL', 'KS', 'AZ', 'MA', 'RI', 'AR', 'IA', 'NC', 'TX', 'CT') then 1 else 0 end as newborn_screening_flag
  from base_table
)
select * from newborn_screening_flag

**Adding Place of Service**

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
with base_table as (
  select distinct *
  from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

tx_patients as (
  select distinct
      patient_id,
      coalesce(rendering_npi, referring_npi) as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where ndc11 in ('54092070001','540920700')
    and service_date between '2023-08-01' and '${end_date}'

  union

  select distinct
      patient_id,
      prescriber_npi as npi,
      fill_date,
      '' as place_of_service
  from com_edp_prd.com_raw.kom_pharmacy_events
  where ndc11 in ('54092070001','540920700')
    and transaction_result = 'PAID'
    and fill_date between '2023-08-01' and '${end_date}'

  union

  select distinct
      patient_id,
      rendering_npi as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
    and service_date between '2023-08-01' and '${end_date}'
),

latest_pos_patients as (
  select
      patient_id,
      nullif(trim(place_of_service), '') as most_recent_infusion_location
  from (
    select
        patient_id,
        place_of_service,
        fill_date,
        row_number() over (
          partition by patient_id
          order by fill_date desc
        ) as rn
    from tx_patients
  ) t
  where rn = 1
)

select
    a.*,
    b.most_recent_infusion_location,
    c.description as most_recent_infusion_description
from base_table a
left join latest_pos_patients b
  on a.patient_id = b.patient_id
left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
  on try_cast(nullif(trim(b.most_recent_infusion_location), '') as int) = c.code



In [0]:
select * from cmpa_insights_internal_schema.patient360_master

In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.patient360 as 
select * from cmpa_insights_internal_schema.patient360_master

### Appendix

In [0]:
-- create or replace table cmpa_insights_internal_schema.patient360 
-- select distinct * from cmpa_insights_internal_schema.patient360

In [0]:
select * from cmpa_insights_internal_schema.patient360

In [0]:
-- create or replace table cmpa_insights_internal_schema.patient360_master
-- select distinct * from cmpa_insights_internal_schema.patient360

In [0]:
select * from cmpa_insights_internal_schema.patient360_master

In [0]:
create or replace view cmpa_insights_internal_schema.patient360 as 
select distinct PATIENT_ID, PATIENT_YOB, PATIENT_AGE, age_bucket, zip3, PATIENT_GENDER, patient_state, incidence_date, first_incidence_treatment_date, latest_claim_date, first_tx_after_diagnosis, latest_mpsii_tx_date, latest_mpsii_tx_type, time_dx_to_first_tx_in_months, treatment_period_months, primary_payer, secondary_payer, active_insurance_group, elaprase_fills, first_dx_hcp_5yr, first_dx_all_visit_count_5yr, first_dx_last_visit_5yr, first_tx_hcp_5yr, first_tx_all_visit_count_5yr, first_tx_treatment_visit_count_5yr, first_tx_last_visit_5yr, most_recently_treated_hcp_2yr, most_recently_treated_hcp_name_2yr, most_recently_treated_hcp_specialty_2yr, most_recently_treated_hcp_secondary_specialty_2yr, most_recently_treated_hcp_2yr_no_of_visits_5yr, most_recently_treated_hcp_2yr_last_visit_5yr, most_recently_treated_hcp_hco_npi, most_recently_treated_hcp_hco_name, PRIMARY_HCP_NPI, primary_hcp_name_2yr, primary_hcp_specialty, primary_hcp_secondary_specialty, SPECIALTY_PRIORITY, NO_OF_VISITS, DX_VISITS, TX_VISITS, MOST_RECENT_VISIT, HCP_RANK, primary_hcp_hco_npi_2yr, primary_hcp_hco_name_2yr, primary_hcp_territory_2yr, primary_hcp_region_2yr, severity, comorbidity_categories, count_of_comorbidity_categories, distinct_comorbidities, count_of_distinct_comorbidities, newborn_screening_flag, most_recent_infusion_location, most_recent_infusion_description
from cmpa_insights_internal_schema.patient360_master

In [0]:
select * from cmpa_insights_internal_schema.patient360
-- select count(*), count(distinct patient_id) from cmpa_insights_internal_schema.patient360

## Appendix

In [0]:

WITH
-- -----------------------------
-- Eligibility (Specified + Incremental Unspecified)
-- -----------------------------
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    UNION ALL
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
),
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND '${end_date}'
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND '${end_date}'
    ) t
),
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)
select * from eligible_patients


In [0]:
-- =============================================================================
-- Patient 360
-- =============================================================================

-- STEP 1: Create MPSII Diagnosis Table
CREATE OR REPLACE TEMPORARY VIEW mpsii_diagnosis_table AS
SELECT * FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODES,
        KH_PLAN_ID AS KH_PLAN,
        Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
        AND TRANSACTION_STATUS = 'PAID'

    UNION

    SELECT DISTINCT 
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        MEDICAL_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODES,
        KH_PLAN_ID AS KH_PLAN,
        Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    SELECT DISTINCT 
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        PHARMACY_EVENT_ID AS EVENT_ID,
        DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
        COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
        NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
        AND TRANSACTION_STATUS = 'PAID'
) AS combined
WHERE FILL_DATE BETWEEN '2020-08-01' AND '${end_date}';

-- STEP 2: Export all patient360 claims for Excel validation
SELECT 
    'DIAGNOSIS' AS claim_type,
    a.PATIENT_ID,
    a.NPI,
    COALESCE(b.FIRST_NAME, '') || ' ' || COALESCE(b.LAST_NAME, '') AS HCP_NAME,
    b.PRIMARY_SPECIALTY,
    b.SECONDARY_SPECIALTY,
    a.FILL_DATE,
    a.DIAGNOSIS_CODES,
    'MEDICAL_EVENTS' AS source,
    NULL AS CODE_TYPE,
    NULL AS NDC11,
    NULL AS PROCEDURE_CODE
FROM mpsii_diagnosis_table a
LEFT JOIN com_edp_prd.com_raw.kom_providers b ON a.NPI = b.NPI
WHERE a.PATIENT_ID IN (SELECT DISTINCT patient_id FROM com_edp_prd.cmpa_insights_internal_schema.patient360)
    AND a.FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'

UNION ALL

SELECT 
    'TREATMENT' AS claim_type,
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    COALESCE(c.FIRST_NAME, '') || ' ' || COALESCE(c.LAST_NAME, '') AS HCP_NAME,
    c.PRIMARY_SPECIALTY,
    c.SECONDARY_SPECIALTY,
    SERVICE_DATE AS FILL_DATE,
    DIAGNOSIS_CODES,
    'MEDICAL_EVENTS' AS source,
    'NDC' AS CODE_TYPE,
    NDC11,
    NULL AS PROCEDURE_CODE
FROM com_edp_prd.com_raw.kom_medical_events
LEFT JOIN com_edp_prd.com_raw.kom_providers c ON COALESCE(RENDERING_NPI, REFERRING_NPI) = c.NPI
WHERE NDC11 IN ('54092070001','540920700')
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT DISTINCT patient_id FROM com_edp_prd.cmpa_insights_internal_schema.patient360)

UNION ALL

SELECT 
    'TREATMENT' AS claim_type,
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    COALESCE(d.FIRST_NAME, '') || ' ' || COALESCE(d.LAST_NAME, '') AS HCP_NAME,
    d.PRIMARY_SPECIALTY,
    d.SECONDARY_SPECIALTY,
    FILL_DATE,
    DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
    'PHARMACY_EVENTS' AS source,
    'NDC' AS CODE_TYPE,
    NDC11,
    NULL AS PROCEDURE_CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
LEFT JOIN com_edp_prd.com_raw.kom_providers d ON PRESCRIBER_NPI = d.NPI
WHERE NDC11 IN ('54092070001','540920700')
    AND TRANSACTION_RESULT = 'PAID'
    AND FILL_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT DISTINCT patient_id FROM com_edp_prd.cmpa_insights_internal_schema.patient360)

UNION ALL

SELECT 
    'TREATMENT' AS claim_type,
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    COALESCE(e.FIRST_NAME, '') || ' ' || COALESCE(e.LAST_NAME, '') AS HCP_NAME,
    e.PRIMARY_SPECIALTY,
    e.SECONDARY_SPECIALTY,
    SERVICE_DATE AS FILL_DATE,
    DIAGNOSIS_CODES,
    'MEDICAL_EVENTS' AS source,
    'PROCEDURE' AS CODE_TYPE,
    NULL AS NDC11,
    PROCEDURE_CODE
FROM com_edp_prd.com_raw.kom_medical_events
LEFT JOIN com_edp_prd.com_raw.kom_providers e ON RENDERING_NPI = e.NPI
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                         '38206','38230','38232','38240','38241','38242','38243','38250')
    AND SERVICE_DATE BETWEEN '2020-08-01' AND '${end_date}'
    AND PATIENT_ID IN (SELECT DISTINCT patient_id FROM com_edp_prd.cmpa_insights_internal_schema.patient360)

ORDER BY PATIENT_ID, FILL_DATE DESC, claim_type;

In [0]:
-- Get all unique specialties from provider table and classify them by priority

WITH all_specialties AS (
    -- Get all unique PRIMARY_SPECIALTY values
    SELECT DISTINCT PRIMARY_SPECIALTY AS specialty_name
    FROM com_edp_prd.com_raw.kom_providers
    WHERE PRIMARY_SPECIALTY IS NOT NULL
    
    UNION
    
    -- Get all unique SECONDARY_SPECIALTY values
    SELECT DISTINCT SECONDARY_SPECIALTY AS specialty_name
    FROM com_edp_prd.com_raw.kom_providers
    WHERE SECONDARY_SPECIALTY IS NOT NULL
),

classified_specialties AS (
    SELECT 
        specialty_name,
        CASE 
            WHEN specialty_name LIKE '%Genetic%' THEN 'Geneticist'
            WHEN specialty_name LIKE '%Psychiatry & Neurology%' OR 
                 specialty_name LIKE '%Neurological Surgery%' OR 
                 specialty_name LIKE '%Neurodevelopmental Disabilities%' THEN 'Psychiatry & Neurology'
            WHEN specialty_name LIKE '%Pediatrics%' THEN 'Pediatrician'
            WHEN specialty_name LIKE '%Internal Medicine%' OR 
                 specialty_name LIKE '%Family Medicine%' THEN 'PCP'
            WHEN specialty_name LIKE '%Nurse Practitioner%' OR 
                 specialty_name LIKE '%Physician Assistant%' THEN 'NPPA'
            ELSE 'Others'
        END AS classified_specialty,
        CASE 
            WHEN specialty_name LIKE '%Genetic%' THEN 1
            WHEN specialty_name LIKE '%Psychiatry & Neurology%' OR 
                 specialty_name LIKE '%Neurological Surgery%' OR 
                 specialty_name LIKE '%Neurodevelopmental Disabilities%' THEN 2
            WHEN specialty_name LIKE '%Pediatrics%' THEN 3
            WHEN specialty_name LIKE '%Internal Medicine%' OR 
                 specialty_name LIKE '%Family Medicine%' THEN 4
            WHEN specialty_name LIKE '%Nurse Practitioner%' OR 
                 specialty_name LIKE '%Physician Assistant%' THEN 5
            ELSE 6
        END AS priority
    FROM all_specialties
)

SELECT 
    specialty_name AS RAW_SPECIALTY,
    classified_specialty AS CLASSIFIED_AS,
    priority AS PRIORITY
FROM classified_specialties
ORDER BY priority ASC, specialty_name ASC;

In [0]:
with all_claims as (
  SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Dx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761','E763')
  AND TRANSACTION_STATUS = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001','540920700')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Tx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001','540920700')
  AND TRANSACTION_RESULT = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                          '38206','38230','38232','38240','38241','38242','38243','38250')
),
relevant_patients as (
  select *
  from all_claims
  where patient_id in (select distinct patient_id from com_edp_prd.cmpa_insights_internal_schema.patient360)
),
patient_geography AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
pulling_relevant_info as (
  select a.patient_id, a.npi, a.fill_date, b.PATIENT_YOB, (year(current_date) - year(b.PATIENT_YOB)) as age, c.PATIENT_STATE, d.FIRST_NAME, d.LAST_NAME, concat(d.FIRST_NAME, ' ', d.LAST_NAME) as hcp_name, d.PRIMARY_SPECIALTY, d.SECONDARY_SPECIALTY, a.claim_type, e.PAYER_NAME, e.INSURANCE_GROUP
  from relevant_patients as a
  left join com_edp_prd.com_raw.kom_patient_demographics as b on a.patient_id = b.PATIENT_ID
  left join patient_geography as c on a.patient_id = c.PATIENT_ID
  left join com_edp_prd.com_raw.kom_providers as d on a.npi = d.npi and d.provider_type = 'INDIVIDUAL'
  left join com_edp_prd.com_raw.kom_plans as e on a.plan_id = e.KH_PLAN_ID
),
poi as (
  select distinct patient_id from pulling_relevant_info
where npi in ('1669821781') and patient_id in (select distinct patient_id from cmpa_insights_internal_schema.patient360_master)
)
select * from pulling_relevant_info where fill_date between "2023-08-01" and "2025-11-30" and patient_id in (select distinct patient_id from poi)

In [0]:
select * 
from cmpa_insights_internal_schema.patient360_master
where PATIENT_ID in ('0VQR3EKL')